# 04 — Full Aero Characterisation

Combine coast-down fits (notebook 02) and ClA estimate (notebook 03) to produce final
CdA, Crr, and active-aero drag-delta values with Monte Carlo uncertainty.
Canada 2026 — straight mode vs corner mode.

In [1]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.aero_params import aggregate_results, extract_crr_and_composite, compute_CdA
from src.uncertainty import propagate_uncertainty, drs_delta_uncertainty

G = 9.81

## Load saved results from notebooks 02 and 03

In [2]:
with open('../results/fit_results.pkl', 'rb') as f:
    fit_data = pickle.load(f)

all_results = fit_data['results']
rho = fit_data['rho']
driver = fit_data['driver']

with open('../results/ClA_estimate.pkl', 'rb') as f:
    cla_data = pickle.load(f)

ClA = cla_data['ClA']
ClA_std = cla_data['ClA_std']

print(f'Driver: {driver}')
print(f'ρ = {rho:.4f} kg/m³')
print(f'Total fit results: {len(all_results)}')
print(f'ClA = {ClA:.3f} ± {ClA_std:.3f} m²')

Driver: 14
ρ = 1.1343 kg/m³
Total fit results: 14
ClA = 3.358 ± 0.649 m²


## Aggregate aero parameters

In [ ]:
params = aggregate_results(all_results, rho, ClA, ClA_std)

print('\n=== Aero Parameter Summary ===')
print(f'Crr                  = {params.Crr:.4f} ± {params.Crr_std:.4f}   (expected 0.015–0.020)')
print(f'ClA                  = {params.ClA:.3f} ± {params.ClA_std:.3f} m²  (expected 2.8–4.5 for 2026)')
print(f'CdA (corner mode)    = {params.CdA_corner_mode:.3f} ± {params.CdA_corner_mode_std:.3f} m²')
print(f'CdA (straight mode)  = {params.CdA_straight_mode:.3f} ± {params.CdA_straight_mode_std:.3f} m²')
print(f'ΔCdA (active aero)   = {params.delta_CdA:.3f} ± {params.delta_CdA_std:.3f} m²  (expected −0.5 to −1.2)')

## Monte Carlo uncertainty propagation

In [ ]:
results_corner   = [r for r in all_results if not r.straight_mode]
results_straight = [r for r in all_results if r.straight_mode]

mc_corner   = propagate_uncertainty(results_corner,   rho, ClA, ClA_std)
mc_straight = propagate_uncertainty(results_straight, rho, ClA, ClA_std) if results_straight else None

print('\n=== Monte Carlo Results ===')
print(f'CdA (corner mode):   {mc_corner.CdA_mean:.3f} ± {mc_corner.CdA_std:.3f} m²'
      f'  [5th–95th: {mc_corner.CdA_p5:.3f}–{mc_corner.CdA_p95:.3f}]')

if mc_straight:
    delta_mean, delta_std = drs_delta_uncertainty(mc_corner, mc_straight)
    print(f'CdA (straight mode): {mc_straight.CdA_mean:.3f} ± {mc_straight.CdA_std:.3f} m²')
    print(f'ΔCdA (active aero):  {delta_mean:.3f} ± {delta_std:.3f} m²')
else:
    delta_mean, delta_std = float('nan'), float('nan')
    print('CdA (straight mode): no straight-mode segments in dataset')

# Alias mc_closed for downstream cells that reference it
mc_closed = mc_corner

## CdA vs lap number — fuel load trend

In [ ]:
corner_results = [r for r in all_results if not r.straight_mode]
if not corner_results:
    print('No corner-mode results to plot.')
else:
    laps_c = [r.lap_number for r in corner_results]
    Crr_c  = [r.beta / (r.m * G) for r in corner_results]
    comp_c = [2.0 * r.alpha / rho for r in corner_results]
    CdA_c  = [compute_CdA(comp, crr, ClA) for comp, crr in zip(comp_c, Crr_c)]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(laps_c, CdA_c, alpha=0.7, s=30, label='corner mode segments')
    ax.axhline(mc_corner.CdA_mean, color='r', linestyle='--', label=f'mean = {mc_corner.CdA_mean:.3f}')
    ax.fill_between(
        [min(laps_c), max(laps_c)],
        mc_corner.CdA_p5, mc_corner.CdA_p95,
        alpha=0.15, color='r', label='5th–95th percentile'
    )
    ax.set_xlabel('Lap number')
    ax.set_ylabel('CdA (m²)')
    ax.set_title('CdA vs lap number (corner mode)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/figures/04_CdA_vs_lap.png', dpi=150)
    plt.show()

## Active aero drag delta — bar chart with uncertainty

In [ ]:
if mc_straight is None:
    print('No straight-mode segments — skipping active aero delta plot.')
else:
    labels = ['Corner mode\n(high downforce)', 'Straight mode\n(low drag)']
    means  = [mc_corner.CdA_mean,   mc_straight.CdA_mean]
    stds   = [mc_corner.CdA_std,    mc_straight.CdA_std]
    colors = ['steelblue', 'tomato']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    bars = ax.bar(labels, means, yerr=stds, capsize=8, color=colors, width=0.5)
    ax.set_ylabel('CdA (m²)')
    ax.set_title('CdA by active aero mode — Canada 2026')
    for bar, val, err in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, val + err + 0.01,
                f'{val:.3f}±{err:.3f}', ha='center', va='bottom', fontsize=9)

    ax = axes[1]
    rng = np.random.default_rng(99)
    delta_samples = (
        rng.normal(mc_straight.CdA_mean, mc_straight.CdA_std, 10_000) -
        rng.normal(mc_corner.CdA_mean,   mc_corner.CdA_std,   10_000)
    )
    ax.hist(delta_samples, bins=50, edgecolor='black', color='orchid')
    ax.axvline(delta_mean, color='r', linestyle='--', label=f'mean = {delta_mean:.3f}')
    ax.axvline(0, color='k', linestyle=':', label='zero')
    ax.set_xlabel('ΔCdA = CdA_straight − CdA_corner (m²)')
    ax.set_ylabel('count')
    ax.set_title('MC distribution of active aero drag delta')
    ax.legend()

    plt.tight_layout()
    plt.savefig('../results/figures/04_active_aero_delta.png', dpi=150)
    plt.show()

    print(f'\nActive aero drag delta: {delta_mean:.3f} ± {delta_std:.3f} m²')
    print(f'Expected (2026 active aero): −0.5 to −1.2 m²')

## Sanity check table

In [ ]:
checks = [
    ('Crr',                   mc_corner.Crr_mean, mc_corner.Crr_std, 0.012, 0.020),
    ('CdA (corner mode)',     mc_corner.CdA_mean, mc_corner.CdA_std, 0.80,  1.20),
    ('ClA',                   ClA,                ClA_std,            2.80,  4.50),
]

print(f'{"Quantity":<30} {"Value":>8}  {"±":>6}  {"Expected":>15}  {"Pass?"}')
print('-' * 78)
for name, val, err, lo, hi in checks:
    in_range = lo <= val <= hi
    flag = '✓' if in_range else '✗  ← CHECK'
    print(f'{name:<30} {val:8.3f}  {err:6.3f}  [{lo:.2f}, {hi:.2f}]     {flag}')

if np.isfinite(delta_mean):
    in_range = -1.20 <= delta_mean <= -0.50
    flag = '✓' if in_range else '✗  ← CHECK'
    print(f'{"ΔCdA (active aero)":<30} {delta_mean:8.3f}  {delta_std:6.3f}  [-1.20, -0.50]     {flag}')
else:
    print(f'{"ΔCdA (active aero)":<30} {"n/a":>8}  {"—":>6}  [-1.20, -0.50]     — no straight-mode data')